# RAG trên PubMedQA (pqa_labeled) - bản gọn
Chạy từ trên xuống. Cell cài thư viện chỉ cần chạy 1 lần, sau đó *Runtime → Restart session*.

## 0. Cài thư viện

In [ ]:
!pip install -q faiss-gpu rank_bm25 datasets google-genai

## 1. Cấu hình thí nghiệm (chỉ sửa ở đây)

In [ ]:
EXP = "exp10"                      # đổi tên mỗi lần thí nghiệm mới
MODEL = "gemini-3.5-flash-lite"
N_QUESTIONS = 200                  # test 20 câu trước, ổn rồi mới lên 200
TOP_K = 5
USE_FEWSHOT = False               # few-shot từng làm kết quả tệ hơn (0.5 vs 0.635) -> mặc định tắt
SLEEP_BETWEEN_CALLS = 6           # giây, để không vượt RPM
OUT_DIR = "/content/drive/MyDrive/rag"
OUT = f"{OUT_DIR}/results_{EXP}.jsonl"

## 2. Import

In [ ]:
import os, json, time, string
import numpy as np
import pandas as pd
import torch
import faiss
import nltk
import re
from nltk.corpus import stopwords
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from google import genai
from google.genai import types
from google.colab import userdata, drive
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 3. Load dataset

In [ ]:
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
print(ds)
print(Counter(ds['train']['final_decision']))

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})
Counter({'yes': 552, 'no': 338, 'maybe': 110})


## 4. Chunking (mỗi context là 1 chunk)

In [ ]:
chunks = []
for ex in ds['train']:
  for contxt in ex['context']['contexts']:
    chunks.append({'pubid': ex['pubid'], 'text': contxt})
print(len(chunks))
chunks_dataset = Dataset.from_list(chunks)

3358


## 5. Dense retrieval: bge-small + FAISS

In [ ]:
model_ckpt = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

def get_embedding(text_list):
  encode_input = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
  encode_input = {k: v.to(device) for k, v in encode_input.items()}
  with torch.no_grad():
    model_output = model(**encode_input)
  return torch.nn.functional.normalize(cls_pooling(model_output), p=2, dim=-1)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
embedding_dataset = chunks_dataset.map(
    lambda x: {'embedding': get_embedding(x['text']).detach().cpu().numpy()}, batched=True)
print(np.array(embedding_dataset["embedding"]).shape)   # mong đợi (3358, 384)
embedding_dataset.add_faiss_index(column="embedding", metric_type=faiss.METRIC_INNER_PRODUCT)

def search(query, k=5):
  question_embedding = get_embedding([query]).cpu().detach().numpy()
  scores, samples = embedding_dataset.get_nearest_examples("embedding", question_embedding, k=k)
  return scores, samples

Map:   0%|          | 0/3358 [00:00<?, ? examples/s]

(3358, 384)


  0%|          | 0/4 [00:00<?, ?it/s]

## 6. Sparse retrieval: BM25 (bỏ dấu câu + stopword)

In [ ]:
def clean_and_tokenize(text):
    text = text.lower()                                               # 1. chữ thường
    text = text.translate(str.maketrans('', '', string.punctuation))  # 2. bỏ dấu câu
    tokens = text.split()                                             # 3. tách từ
    return [w for w in tokens if w not in stop_words]                 # 4. bỏ stopword

corpus_tokens = [clean_and_tokenize(t) for t in chunks_dataset['text']]
bm25 = BM25Okapi(corpus_tokens)

def search_bm25(query, k=5):
  query_tokens = clean_and_tokenize(query)   # tokenize query giống hệt corpus
  scores = bm25.get_scores(query_tokens)
  top_indices = np.argsort(scores)[::-1][:k]
  return scores[top_indices], chunks_dataset.select(top_indices)

## 7. Đánh giá retrieval: Recall@k và MRR (không tốn quota)

In [ ]:
def eval_retrieval(search_fn, n=200, k=5, show_wrong=False):
  hits, mrr = 0, 0
  for i in range(n):
    ex = ds['train'][i]
    scores, samples = search_fn(ex['question'], k)
    for j, p in enumerate(samples['pubid']):
      if p == ex['pubid']:
        hits += 1
        mrr += 1/(j+1)
        break
    else:
      if show_wrong:
        print(ex['question'], ex['pubid'], samples['pubid'])
  print(f"Recall@{k} = {hits/n:.3f}   MRR = {mrr/n:.3f}")

eval_retrieval(search)        # dense
eval_retrieval(search_bm25)   # BM25

Recall@5 = 0.985   MRR = 0.968
Recall@5 = 0.960   MRR = 0.938


## 8. Prompt
`build_context` đánh số passage `[i] (pubid X) text`. Few-shot chỉ dùng khi `USE_FEWSHOT = True`.

In [ ]:
def build_context(samples):
    lines = []
    for i, (pid, text) in enumerate(zip(samples["pubid"], samples["text"]), start=1):
        lines.append(f"[{i}] (pubid {pid}) {text}")
    return "\n\n".join(lines)

In [ ]:
# ---- (TÙY CHỌN) Few-shot: lấy ví dụ từ câu 200+ để không trùng tập test ----
def make_example(ex, n_ctx=2, max_chars=400):
    ctxs = ex['context']['contexts'][-n_ctx:]   # đoạn cuối thường là RESULTS
    passages = "\n".join(f"[{i+1}] {c[:max_chars]}" for i, c in enumerate(ctxs))
    return f"Passages:\n{passages}\nQuestion: {ex['question']}\nAnswer:\n{ex['final_decision']}\n{len(ctxs)}"

pick = lambda lab: next(ds['train'][i] for i in range(200, 1000) if ds['train'][i]['final_decision'] == lab)
example_no, example_maybe, example_yes = (make_example(pick(l)) for l in ("no", "maybe", "yes"))

#Vì few-shot không thật sự giúp ích qua các lần chạy thử nghiệm nên ta sẽ bỏ hẳn

In [ ]:
RULES = """Read the passages and answer the question.

Rules:
- "yes": the authors conclude the effect/association exists.
- "no": the authors conclude there is no effect, or no significant difference.
- "maybe": the authors explicitly say results are mixed, inconclusive, or need further study.
- The passages contain results but not the authors' final conclusion. Infer the answer from the results; do NOT answer maybe just because a conclusion is not stated.
- Small, non-significant, or no differences → no.
"""

FEWSHOT = f"""
EXAMPLES:
### Example 1
{example_no}
### Example 2
{example_maybe}
### Example 3
{example_yes}

"""

QUERY = """

Passages:
{context}

Question: {question}

Respond in exactly this format:
Conclusion: <one sentence: based on the RESULTS in the passages, what is the answer to the question?>
Answer: <yes|no|maybe>
Sources: <passage numbers>
"""

PROMPT = RULES + (FEWSHOT if USE_FEWSHOT else "") + QUERY

# Kiểm tra prompt cho câu 0 trước khi gọi API
q = ds['train'][0]['question']
scores, samples = search(q, k=TOP_K)
print(PROMPT.format(question=q, context=build_context(samples)))

Read the passages and answer the question.

Rules:
- "yes": the authors conclude the effect/association exists.
- "no": the authors conclude there is no effect, or no significant difference.
- "maybe": the authors explicitly say results are mixed, inconclusive, or need further study.
- The passages contain results but not the authors' final conclusion. Infer the answer from the results; do NOT answer maybe just because a conclusion is not stated.
- Small, non-significant, or no differences → no.


Passages:
[1] (pubid 21645374) The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos and exa

## 9. Gemini client + gọi có retry

In [ ]:
# Key lưu trong Colab Secrets (biểu tượng chìa khóa), KHÔNG dán key vào code
client = genai.Client(api_key=userdata.get("API-key"),
                      http_options=types.HttpOptions(timeout=30000))

def call_with_retry(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(model=MODEL, contents=prompt)
            return resp.text or ""
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("429 - chờ 20s rồi thử lại...")
                time.sleep(20)
            else:
                print("Lỗi:", e)
                time.sleep(5)
    raise RuntimeError("Hết số lần retry (có thể đã hết quota ngày)")

def answer(question, k=TOP_K):
    scores, samples = search(question, k)
    prompt = PROMPT.format(question=question, context=build_context(samples))
    return call_with_retry(prompt)

## 10. Chạy thí nghiệm (có resume)
Câu nào đã có trong `OUT` sẽ được bỏ qua, nên Colab mất kết nối thì chạy lại cell này là được, không tốn request lặp.

In [ ]:
drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        done = {json.loads(line)["pubid"] for line in f if line.strip()}
print(f"Đã có {len(done)} câu trong {OUT}")

for row in ds['train'].select(range(N_QUESTIONS)):
  if row["pubid"] in done:
    continue
  raw = answer(row['question'])
  m = re.search(r"Answer:\s*(yes|no|maybe)", raw, re.I)
  pred = m.group(1).lower() if m else "parse_fail"
  with open(OUT, "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "pubid": row["pubid"],
        "question": row["question"],
        "gold": row['final_decision'],
        "pred": pred,
        "raw": raw,   # output gốc chưa cắt
    }, ensure_ascii=False) + "\n")
  done.add(row["pubid"])
  time.sleep(SLEEP_BETWEEN_CALLS)
print("Xong")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã có 20 câu trong /content/drive/MyDrive/rag/results_exp10.jsonl
Xong


## 11. Đánh giá

In [ ]:
df = pd.read_json(OUT, lines=True).drop_duplicates("pubid", keep="first")
golds, preds = df["gold"].tolist(), df["pred"].tolist()
print("n =", len(df))
print("accuracy:", accuracy_score(golds, preds))
print("macro F1:", f1_score(golds, preds, average="macro"))
print(classification_report(golds, preds, zero_division=0))
print(pd.crosstab(df["gold"], df["pred"], margins=True))

n = 200
accuracy: 0.67
macro F1: 0.5759585824803216
              precision    recall  f1-score   support

       maybe       0.26      0.23      0.24        35
          no       0.73      0.70      0.71        53
         yes       0.75      0.79      0.77       112

    accuracy                           0.67       200
   macro avg       0.58      0.57      0.58       200
weighted avg       0.66      0.67      0.66       200

pred   maybe  no  yes  All
gold                      
maybe      8   9   18   35
no         5  37   11   53
yes       18   5   89  112
All       31  51  118  200


# Thay cách retrieval bằng Hybrid Search

In [ ]:
def hybrid_search(query, k=TOP_K, n=20, c=60, w_dense=0.7, w_bm25=0.3):
    # dense: lấy index dòng
    q_emb = get_embedding([query]).cpu().detach().numpy()
    _, dense_idx = embedding_dataset.search("embedding", q_emb[0], k=n)
    # bm25: lấy index dòng
    bm_scores = bm25.get_scores(clean_and_tokenize(query))
    sparse_idx = np.argsort(bm_scores)[::-1][:n]

    score = {}
    for lst, w in ((dense_idx, w_dense), (sparse_idx, w_bm25)):
        for rank, idx in enumerate(lst):
            idx = int(idx)
            score[idx] = score.get(idx, 0) + w / (c + rank + 1)
    top = sorted(score, key=score.get, reverse=True)[:k]
    return [score[i] for i in top], chunks_dataset.select(top)[:]

In [ ]:
for wd, wb in [(1.0, 1.0), (0.7, 0.3), (0.3, 0.7)]:
    print(f"dense:bm25 = {wd}:{wb}")
    eval_retrieval(lambda q, k=5: hybrid_search(q, k=k, w_dense=wd, w_bm25=wb))

dense:bm25 = 1.0:1.0
Recall@5 = 0.980   MRR = 0.965
dense:bm25 = 0.7:0.3
Recall@5 = 0.985   MRR = 0.968
dense:bm25 = 0.3:0.7
Recall@5 = 0.980   MRR = 0.961


## Kết luận Hybrid Search (Ngày 5)

| Retriever | Recall@5 | MRR |
|---|---|---|
| Dense (bge-small) | **0.985** | **0.968** |
| BM25 (clean tokenize) | 0.960 | 0.938 |
| Hybrid RRF 1:1 | 0.980 | 0.965 |
| Hybrid RRF 0.7:0.3 | ≈ dense | ≈ dense |
| Hybrid RRF 0.3:0.7 | ≤ dense | ≤ dense |

**Kết luận:** Hybrid RRF **không cải thiện** so với dense trên PubMedQA `pqa_labeled` (200 câu đầu).

**Lý do:**
1. Dense đã gần chạm trần (197/200 câu đúng ở top-5), nên BM25 gần như không còn chỗ để bù.
2. Câu hỏi PubMedQA thường được viết lại từ tiêu đề/abstract, nên embedding bắt rất tốt; lợi thế từ khóa exact-match của BM25 ít phát huy.
3. Khi gộp 1:1, BM25 (yếu hơn) còn kéo hybrid xuống nhẹ (trượt thêm ~1 câu so với dense).

**Quyết định cho pipeline:** giữ **dense top-5** làm retriever chính. Hybrid vẫn giữ trong notebook như ablation để chứng minh đã thử. Không chạy LLM lại với hybrid (đỡ tốn quota), vì retrieval không tốt hơn thì end-to-end khó hơn exp10 (acc 0.67).

Tham chiếu: Cormack et al., *Reciprocal Rank Fusion*, 2009; Jin et al., *PubMedQA*, 2019.

# Ta thử với các k lớn hơn

In [ ]:
for k in (3, 5, 8, 10):
    print(f"--- dense k={k}")
    eval_retrieval(search, n=200, k=k)

--- dense k=3
Recall@3 = 0.980   MRR = 0.967
--- dense k=5
Recall@5 = 0.985   MRR = 0.968
--- dense k=8
Recall@8 = 0.990   MRR = 0.969
--- dense k=10
Recall@10 = 0.990   MRR = 0.969


## Ablation TOP_K (generation, Ngày 5)

Cùng prompt exp10 (Conclusion suy từ RESULTS) + dense bge-small + Gemini 3.5 Flash Lite. Đo end-to-end trên **200 câu đầu** `pqa_labeled` (trừ bản smoke 20 câu lúc chọn k).

### Smoke trên 20 câu đầu
| k | accuracy | macro-F1 |
|---|---|---|
| 3 | 0.700 | 0.533 |
| 5 | 0.700 | 0.533 |
| 8 | 0.700 | 0.533 |
| 10 | 0.750 | 0.549 |

→ Chênh 1 câu; mẫu nhỏ chưa đủ để chọn k.

### Full 200 câu
| k | accuracy | macro-F1 | yes F1 | no F1 | maybe F1 |
|---|---|---|---|---|---|
| 5 (exp10) | 0.670 | 0.576 | 0.77 | 0.71 | 0.24 |
| 8 | 0.655 | 0.555 | 0.77 | 0.69 | 0.21 |
| **10** | **0.700** | **0.598** | **0.80** | **0.73** | **0.27** |

Retrieval (dense) chỉ tăng nhẹ khi tăng k: Recall@5 = 0.985 → Recall@10 = 0.990. End-to-end thì k=10 tốt hơn rõ; k=8 **thấp hơn** cả k=5 (nhiễu passage / variance).

**Quyết định:** chốt `TOP_K = 10` cho pipeline. Hybrid RRF không dùng production (xem cell kết luận hybrid phía trên).